In [8]:
import pickle
import torch
import torch.nn as nn
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.models import ERModel
from pykeen.nn import DistMultInteraction, TransEInteraction, RotatEInteraction
from pykeen.models import DistMult, CompGCN, NodePiece
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator, SampledRankBasedEvaluator
from torch.optim import Adam, RMSprop, NAdam


In [9]:
main_data = pd.read_csv('data/edges/triples.csv')
main_data = main_data.astype(str)

triples = main_data[['id_entity_1', 'predicate', 'id_entity_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples, create_inverse_triples=True)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=17)



In [ ]:
EMB_DIM = 64
MARGIN = 1.1
LR = 1e-3
NUM_NEGS_PER_POS = 15
EPOCHS = 2
BATCH_SIZE = 1024
WEIGHT = 1e-3
device = "cuda"

# dropout settings for CompGCN layer
GCN_DROPOUT = 0.1              # CompGCNLayer.dropout (forward/backward edges)
ATTN_HEADS = 4                 # CompGCNLayer.attention_heads (only if edge_weighting uses attention)
ATTN_DROPOUT = 0.1             # CompGCNLayer.attention_dropout (only if attention edge weighting is used)

loss_function = MarginRankingLoss(margin=MARGIN)

model = CompGCN(
    triples_factory=training_set,
    embedding_dim=EMB_DIM,
    random_seed=100,
    loss=loss_function,
    interaction="distmult",
    encoder_kwargs=dict(
        num_layers=2,
        dims=EMB_DIM,
        layer_kwargs=dict(
            dropout=GCN_DROPOUT,
            activation=nn.ReLU,
            # если хочешь attention-взвешивание сообщений, раскомментируй:
            # edge_weighting="attention",
            # attention_heads=ATTN_HEADS,
            # attention_dropout=ATTN_DROPOUT,
        ),
    ),
).to(device)

optimizer = Adam(params=model.get_grad_params(), lr=LR, weight_decay=WEIGHT)

training_loop = SLCWATrainingLoop(
    model=model,
    triples_factory=training_set,
    optimizer=optimizer,
    negative_sampler="pseudotyped",
    negative_sampler_kwargs=dict(num_negs_per_pos=NUM_NEGS_PER_POS),
)

evaluator = RankBasedEvaluator()

training_loop.train(
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    triples_factory=training_set,
    use_tqdm_batch=False,
)

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples[:1000].to(device),
    additional_filter_triples=[
        training_set.mapped_triples.to(device),
        validation_set.mapped_triples.to(device),
    ],
)

metrics = model_results.to_df()
metrics = metrics[(metrics["Side"] == "both") & (metrics["Rank_type"] == "realistic")]
metrics